# Project 1 — Data Cleaning & Preparation

## Phase 2: Data Quality Audit

The purpose of this phase is to examine the raw dataset for data-quality issues before making any cleaning changes.

The audit will focus on missing values, duplicate records, duplicate identifiers, date formats, numerical values, text consistency, and other basic data-integrity checks.

In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("../data/raw/Dataset for Data Analytics.xlsx")

In [3]:
df.shape

(1200, 14)

## 2.1 Missing Value Audit

The first audit checks for missing or null values across all columns. At this stage, the purpose is only to identify and measure missing values. No values will be replaced or removed yet.

In [4]:
df.isnull().sum()

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

In [5]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary

,missing_count,missing_percentage
OrderID,0,0.00
Date,0,0.00
CustomerID,0,0.00
Product,0,0.00
Quantity,0,0.00
UnitPrice,0,0.00
ShippingAddress,0,0.00
PaymentMethod,0,0.00
OrderStatus,0,0.00
TrackingNumber,0,0.00


In [6]:
missing_summary.sort_values(
    by="missing_count",
    ascending=False
)

,missing_count,missing_percentage
CouponCode,309,25.75
OrderID,0,0.00
CustomerID,0,0.00
Product,0,0.00
Quantity,0,0.00
Date,0,0.00
UnitPrice,0,0.00
ShippingAddress,0,0.00
OrderStatus,0,0.00
PaymentMethod,0,0.00


### Initial Finding

The missing-value audit shows that `CouponCode` contains missing values, while the remaining columns do not show missing values in the initial audit.

The missing `CouponCode` values will not be changed at this stage. The reason for the missing values will be considered before deciding on an appropriate treatment.

## 2.2 Inspecting Missing Values

The missing-value summary identified the columns containing null values. I will now inspect the affected records to understand whether the missing values appear to represent a data-entry issue or a meaningful absence of information.

In [7]:
df[df["CouponCode"].isnull()].head(10)

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
8,ORD200008,2025-04-02,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,NaN,Email,268.56
15,ORD200015,2023-07-17,C39416,Printer,1,473.96,942 Main St,Cash,Delivered,TRK54930938,3,NaN,Google,473.96
17,ORD200017,2024-03-02,C60672,Tablet,1,423.40,382 Main St,Debit Card,Shipped,TRK11972116,3,NaN,Referral,423.40
21,ORD200021,2024-11-17,C99023,Monitor,4,342.95,569 Main St,Cash,Delivered,TRK52646983,6,NaN,Google,1371.80
26,ORD200026,2024-03-06,C21944,Laptop,5,59.55,296 Main St,Online,Delivered,TRK80276812,6,NaN,Referral,297.75
28,ORD200028,2023-11-21,C59823,Phone,4,160.07,568 Main St,Online,Pending,TRK65895539,5,NaN,Email,640.28
31,ORD200031,2023-06-30,C26445,Chair,3,500.08,996 Main St,Debit Card,Returned,TRK31229981,6,NaN,Facebook,1500.24
32,ORD200032,2023-05-01,C12388,Tablet,5,536.72,830 Main St,Cash,Delivered,TRK99261395,10,NaN,Email,2683.60
36,ORD200036,2025-05-08,C59337,Laptop,2,534.43,465 Main St,Debit Card,Pending,TRK50961250,5,NaN,Facebook,1068.86
42,ORD200042,2024-05-06,C28712,Printer,2,151.66,980 Main St,Cash,Cancelled,TRK40772558,6,NaN,Instagram,303.32


In [8]:
df[df["CouponCode"].isnull()][
    ["OrderID", "Product", "Quantity", "UnitPrice", "CouponCode", "TotalPrice"]
].head(10)

,OrderID,Product,Quantity,UnitPrice,CouponCode,TotalPrice
8,ORD200008,Phone,2,134.28,NaN,268.56
15,ORD200015,Printer,1,473.96,NaN,473.96
17,ORD200017,Tablet,1,423.40,NaN,423.40
21,ORD200021,Monitor,4,342.95,NaN,1371.80
26,ORD200026,Laptop,5,59.55,NaN,297.75
28,ORD200028,Phone,4,160.07,NaN,640.28
31,ORD200031,Chair,3,500.08,NaN,1500.24
32,ORD200032,Tablet,5,536.72,NaN,2683.60
36,ORD200036,Laptop,2,534.43,NaN,1068.86
42,ORD200042,Printer,2,151.66,NaN,303.32


In [9]:
df["CouponCode"].isnull().sum()

np.int64(309)

### Missing Value Finding

The `CouponCode` field contains missing values. The affected records will be retained during the audit because the reason for the missing value has not yet been established. The appropriate treatment will be decided during the cleaning phase after considering the meaning of the field.

## 2.3 Duplicate Row Audit

The next audit checks whether any complete records are duplicated in the raw dataset.

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df[df.duplicated(keep=False)]

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice


### Duplicate Row Finding

The complete-row duplicate check was performed to identify records that are repeated across all columns. Any duplicates found will be investigated before deciding whether removal is appropriate.

## 2.4 Duplicate OrderID Audit

Since `OrderID` is used to identify an order, I will check whether any OrderID appears more than once.

In [12]:
df["OrderID"].duplicated().sum()

np.int64(0)

In [13]:
df[df["OrderID"].duplicated(keep=False)].sort_values("OrderID")

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice


In [14]:
duplicate_order_ids = (
    df.groupby("OrderID")
      .size()
      .reset_index(name="count")
      .query("count > 1")
)

duplicate_order_ids

,OrderID,count


### OrderID Finding

The `OrderID` field was checked for repeated identifiers. The result of this check will be used as part of the final data-integrity validation.

## 2.5 Date Quality Audit

The `Date` column will be checked for its current data type, missing or invalid values, and the overall date range.

In [15]:
df["Date"].dtype

dtype('<M8[ns]')

In [16]:
df["Date"].isnull().sum()

np.int64(0)

In [17]:
converted_dates = pd.to_datetime(df["Date"], errors="coerce")

converted_dates.isnull().sum()

np.int64(0)

In [18]:
df["Date"].min(), df["Date"].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2025-06-30 00:00:00'))

In [19]:
df["Date"].sort_values().head()

112   2023-01-01
848   2023-01-01
236   2023-01-01
373   2023-01-02
645   2023-01-02
Name: Date, dtype: datetime64[ns]

In [20]:
df["Date"].sort_values().tail()

176    2025-06-28
1039   2025-06-28
773    2025-06-28
1107   2025-06-30
256    2025-06-30
Name: Date, dtype: datetime64[ns]

### Date Finding

The `Date` column was reviewed for datatype consistency, missing values, conversion failures, and the date range represented in the raw dataset.

## 2.6 Numeric Data Audit

The numerical fields will be reviewed for their data types, ranges, and values that could indicate data-quality problems.

In [21]:
df[["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]].dtypes

Quantity         int64
UnitPrice      float64
ItemsInCart      int64
TotalPrice     float64
dtype: object

In [22]:
df[["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]].describe()

,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200.000000,1200.000000,1200.000000,1200.000000
mean,2.945833,356.412750,5.485000,1053.968300
std,1.407557,197.177146,2.281983,819.856558
min,1.000000,11.390000,1.000000,11.390000
25%,2.000000,186.062500,4.000000,410.520000
50%,3.000000,364.210000,5.000000,823.615000
75%,4.000000,521.570000,7.000000,1578.475000
max,5.000000,699.930000,10.000000,3456.400000


In [23]:
df[df["Quantity"] <= 0]

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice


In [24]:
df[df["UnitPrice"] <= 0]

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice


In [25]:
df[df["ItemsInCart"] < 0]

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice


In [26]:
df[df["TotalPrice"] < 0]

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice


### Numeric Finding

The numeric fields were stored using appropriate numeric data types. No invalid quantity, unit price, item count, or total price values were identified during the audit.

## 2.7 Text Consistency Audit

The main categorical fields will be reviewed for inconsistent spelling, capitalization, whitespace, or unexpected categories.

In [27]:
df["Product"].value_counts(dropna=False)

Product
Printer    181
Tablet     179
Chair      178
Laptop     173
Desk       170
Monitor    163
Phone      156
Name: count, dtype: int64

In [28]:
df["PaymentMethod"].value_counts(dropna=False)

PaymentMethod
Online         258
Cash           246
Credit Card    234
Debit Card     232
Gift Card      230
Name: count, dtype: int64

In [29]:
df["OrderStatus"].value_counts(dropna=False)

OrderStatus
Cancelled    250
Returned     247
Pending      237
Shipped      235
Delivered    231
Name: count, dtype: int64

In [30]:
df["ReferralSource"].value_counts(dropna=False)

ReferralSource
Instagram    259
Email        250
Google       241
Facebook     228
Referral     222
Name: count, dtype: int64

In [31]:
df["CouponCode"].value_counts(dropna=False)

CouponCode
FREESHIP    313
NaN         309
WINTER15    292
SAVE10      286
Name: count, dtype: int64

In [32]:
text_columns = [
    "Product",
    "PaymentMethod",
    "OrderStatus",
    "CouponCode",
    "ReferralSource"
]

whitespace_check = {}

for column in text_columns:
    whitespace_check[column] = (
        df[column].dropna()
        .astype(str)
        .str.strip()
        .ne(df[column].dropna().astype(str))
        .sum()
    )

whitespace_check

{'Product': np.int64(0),
 'PaymentMethod': np.int64(0),
 'OrderStatus': np.int64(0),
 'CouponCode': np.int64(0),
 'ReferralSource': np.int64(0)}

### Text Consistency Finding

The main categorical fields were reviewed for their observed values and leading or trailing whitespace. No obvious inconsistent text values or whitespace issues were identified during the audit.

## 2.8 Identifier Format Audit

The identifier fields will be checked to determine whether their values follow the expected structure used throughout the dataset.

In [33]:
order_id_check = df["OrderID"].str.match(r"^ORD\d{6}$", na=False)

order_id_check.value_counts()

OrderID
True    1200
Name: count, dtype: int64

In [34]:
df.loc[~order_id_check, ["OrderID"]]

,OrderID


In [35]:
customer_id_check = df["CustomerID"].str.match(r"^C\d{5}$", na=False)

customer_id_check.value_counts()

CustomerID
True    1200
Name: count, dtype: int64

In [36]:
df.loc[~customer_id_check, ["CustomerID"]]

,CustomerID


In [37]:
tracking_check = df["TrackingNumber"].str.match(r"^TRK\d{8}$", na=False)

tracking_check.value_counts()

TrackingNumber
True    1200
Name: count, dtype: int64

In [38]:
df.loc[~tracking_check, ["TrackingNumber"]]

,TrackingNumber


### Identifier Finding

All OrderID, CustomerID, and TrackingNumber values matched the observed identifier formats. No invalid identifier formats were identified.

## 2.9 Logical and Business Rule Checks

The dataset contains related numerical fields that can be compared against simple business rules. The main check will compare `TotalPrice` with `Quantity × UnitPrice`.

In [39]:
expected_total = (
    df["Quantity"] * df["UnitPrice"]
).round(2)

expected_total.head()

0    2853.10
1     302.70
2    2753.40
3     273.19
4    2504.04
dtype: float64

In [40]:
total_price_check = (
    expected_total == df["TotalPrice"].round(2)
)

total_price_check.value_counts()

True    1200
Name: count, dtype: int64

In [41]:
df.loc[
    ~total_price_check,
    ["OrderID", "Quantity", "UnitPrice", "TotalPrice"]
]

,OrderID,Quantity,UnitPrice,TotalPrice


### Business Rule Finding

All 1,200 records passed the `TotalPrice = Quantity × UnitPrice` check. No mismatches were identified.

## 2.10 Audit Summary

The data-quality audit has now covered missing values, duplicate records, identifiers, dates, numerical fields, text consistency, and basic logical relationships between fields.

The findings below will be used to decide which cleaning actions are required in the next phase.

In [42]:
audit_summary = pd.DataFrame({
    "Check": [
        "Missing values",
        "Duplicate rows",
        "Duplicate OrderIDs",
        "Missing dates",
        "Invalid OrderID formats",
        "Invalid CustomerID formats",
        "Invalid TrackingNumber formats",
        "TotalPrice mismatches"
    ],
    "Result": [
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        df["OrderID"].duplicated().sum(),
        df["Date"].isnull().sum(),
        (~order_id_check).sum(),
        (~customer_id_check).sum(),
        (~tracking_check).sum(),
        (~total_price_check).sum()
    ]
})

audit_summary

,Check,Result
0,Missing values,309
1,Duplicate rows,0
2,Duplicate OrderIDs,0
3,Missing dates,0
4,Invalid OrderID formats,0
5,Invalid CustomerID formats,0
6,Invalid TrackingNumber formats,0
7,TotalPrice mismatches,0


### Overall Audit Finding

The raw dataset has been systematically reviewed for the main data-quality issues identified in the project requirements. The audit results provide the evidence needed to determine which fields require cleaning and how those issues should be handled in the next phase.